# Préparation des données - Projet Churn

## 1. Contexte
Suite à l'Analyse Exploratoire des Données (EDA), ce notebook se concentre sur la préparation technique des données pour la modélisation. L'objectif est de transformer le dataset brut en un format exploitable par nos algorithmes de Machine Learning.

**Objectifs :**
- Chargement des données et contrôle qualité (QC).
- Nettoyage des colonnes non pertinentes.
- Création de variables dérivées (Feature Engineering).
- Division du dataset (Train/Test Split).
- Construction d'un pipeline de prétraitement reproductible.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Contrôle Qualité (QC) et Chargement
Dans cette phase, nous chargeons les données et vérifions leur intégrité (valeurs manquantes, doublons).

In [ ]:
def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data' / 'customer_churn.csv').exists():
            return candidate
    raise FileNotFoundError('Impossible de localiser la racine du projet contenant data/customer_churn.csv')

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / 'data' / 'customer_churn.csv'

df = pd.read_csv(DATA_PATH)
print(f'Dimensions initiales : {df.shape}')

# Vérification des valeurs manquantes
print('\nValeurs manquantes par colonne :')
print(df.isna().sum().sort_values(ascending=False).head(10))

# Gestion des doublons
print(f'\nNombre de doublons : {df.duplicated().sum()}')
df = df.drop_duplicates().reset_index(drop=True)

# Suppression des colonnes techniques non prédictives
if 'customer_id' in df.columns:
    df = df.drop(columns=['customer_id'])
    print("\nColonne 'customer_id' supprimée.")

print(f'Dimensions après QC : {df.shape}')

## 3. Split des données
Nous séparons la variable cible (`churn`) des variables explicatives, puis nous divisons le dataset en ensembles d'entraînement et de test.

In [ ]:
target_col = 'churn'
if target_col not in df.columns:
    raise KeyError(f'La colonne cible {target_col} est absente du dataset.')

# Séparation X/y
X = df.drop(columns=[target_col])
y = df[target_col]

# Split Train/Test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f'Taille X_train : {X_train.shape}')
print(f'Taille X_test : {X_test.shape}')

## 4. Pipeline de Preprocessing
Nous appliquons les transformations nécessaires : 
- **Feature Engineering** : calcul de ratios pertinents.
- **Variables numériques** : Imputation (médiane) et Standardisation.
- **Variables catégorielles** : Imputation (plus fréquente) et Encodage One-Hot.

In [ ]:
# Feature Engineering sur X_train et X_test
def apply_feature_engineering(data):
    temp_df = data.copy()
    if {'support_tickets', 'tenure_months'}.issubset(temp_df.columns):
        tenure_safe = temp_df['tenure_months'].replace(0, np.nan)
        temp_df['tickets_per_tenure'] = (temp_df['support_tickets'] / tenure_safe).fillna(0)
    
    if {'total_revenue', 'tenure_months'}.issubset(temp_df.columns):
        tenure_safe = temp_df['tenure_months'].replace(0, np.nan)
        temp_df['revenue_per_month'] = (temp_df['total_revenue'] / tenure_safe).fillna(0)
        
    return temp_df

X_train = apply_feature_engineering(X_train)
X_test = apply_feature_engineering(X_test)

print('Features après engineering :', X_train.columns.tolist())

In [ ]:
# Identification des colonnes par type
num_cols = X_train.select_dtypes(include=['number', 'bool']).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=['number', 'bool']).columns.tolist()

# Pipelines individuels
num_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Assemblage via ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

print(f'Colonnes numériques : {len(num_cols)}')
print(f'Colonnes catégorielles : {len(cat_cols)}')

print("\nLe preprocessor est prêt pour l'entraînement.")

## Conclusion du Preprocessing
Le dataset est maintenant propre, divisé et les pipelines de transformation sont définis. 

**Étape suivante :** Entraînement des modèles dans le notebook `03_train.ipynb`.